# Per-meet engagement
Drill into a single meet day. Set `MEET_ID` and `PI_LOCAL_DATE` below.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('_lib'))
from engagement_query import run_kql, per_meet_query, workspace_id_from_env
import pandas as pd

MEET_ID = 'replace-me'
PI_LOCAL_DATE = '2026-03-28'
WS = workspace_id_from_env()

In [ ]:
df = run_kql(WS, per_meet_query(MEET_ID, PI_LOCAL_DATE), days=14)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.head()

## Viewing sessions (5-min gap rule)

In [ ]:
sessions = (df.groupby(['viewer_id', 'session_idx'])
             .agg(start=('timestamp', 'min'),
                  end=('timestamp', 'max'),
                  events=('ev', 'count'))
             .reset_index())
sessions['duration_s'] = (sessions['end'] - sessions['start']).dt.total_seconds()
sessions.describe()

## Drop-off by event type

In [ ]:
views = df[df['ev'] == 'viewer_event_view']
drop = (views.groupby(['stroke_name', 'age_group', 'gender'])
             .agg(unique_viewers=('viewer_id', 'nunique'),
                  views=('viewer_id', 'count'))
             .reset_index()
             .sort_values('unique_viewers', ascending=False))
drop.head(30)

## Concurrent-viewer peak (1-minute buckets)

In [ ]:
hb = df[df['ev'] == 'viewer_heartbeat'].copy()
hb['bucket'] = hb['timestamp'].dt.floor('1min')
peak = hb.groupby('bucket')['viewer_id'].nunique()
peak.plot(title='Concurrent viewers per minute');

## LCP and connection-type breakdown

In [ ]:
loads = df[df['ev'] == 'viewer_page_load']
loads['effective_type'].value_counts()

In [ ]:
lcp = df[df['ev'].isin(['viewer_hidden', 'viewer_unload'])]['lcp_ms']
lcp = lcp[lcp > 0]
lcp.describe()